In [0]:
from util import get_all_energy_type, to_tabular_format
from cwarler import single_energy_record

energy_type = get_all_energy_type("Coal")
data = single_energy_record(energy_type['_id'])


In [0]:
# #  Convert Data to tabular formate
df = to_tabular_format(data['data'],energy_type["_name"])
print(df.head(2))


In [0]:
catalog_name = "jeragm"
schema_name = "bronze_layer"
table_name = "ing_china_nbs_output_of_energy_products_current_period_actuals_monthly"


In [0]:
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog_name}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog_name}.{schema_name}")

In [0]:
# Step 5: Convert pandas -> Spark DataFrame
sdf = spark.createDataFrame(df)


In [0]:
sdf.printSchema()

In [0]:
from pyspark.sql.functions import col

# -----------------------------
# Step 1: Define important columns (keep even if null)
# -----------------------------
important_cols = ["kj1_name", "kj2_name", "kj3_name"]

# -----------------------------
# Step 2: Convert important void columns → string
# -----------------------------
for field in sdf.schema.fields:
    if field.name in important_cols and field.dataType.typeName() == "void":
        sdf = sdf.withColumn(field.name, col(field.name).cast("string"))

# -----------------------------
# Step 3: Identify remaining void columns
# -----------------------------
void_cols = [
    field.name
    for field in sdf.schema.fields
    if field.dataType.typeName() == "void" and field.name not in important_cols
]

# -----------------------------
# Step 4: Drop unwanted void columns
# -----------------------------
if void_cols:
    print("Dropping void columns:", void_cols)
    sdf = sdf.drop(*void_cols)

# -----------------------------
# Step 5: Final schema validation (important!)
# -----------------------------
for field in sdf.schema.fields:
    if field.dataType.typeName() == "void":
        raise Exception(f"❌ Still found void column: {field.name}")

print("✅ Schema is clean. No void columns remaining.")

# -----------------------------
# Step 6: Save into Unity Catalog Delta table
# -----------------------------
sdf.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.{schema_name}.{table_name}")

print(f"✅ Saved successfully to {catalog_name}.{schema_name}.{table_name}")

In [0]:
sdf.head(5)